In [3]:
import pandas as pd
import numpy as np

# 1. Carga del dataset desde el archivo Excel
df = pd.read_excel('informacion_proyectos_plan_6gw_plus.xlsx')

# 2. Exploración (EDA) y Diagnóstico
print(df.isna().sum()) # Detección de valores nulos
print(df.duplicated().sum()) # Detección de filas redundantes

print("DataFrame Original:")
print(df)
print("Información del DataFrame:")
df.info()
print("Valores nulos por columna:")
print(df.isnull().sum())

# 3. Pipeline de Transformación
df_clean = df.copy()

# A. Limpieza Estructural
# Eliminar la columna 'Index' que viene quemada en el Excel, ya que pandas genera su propio índice.
if 'Index' in df_clean.columns:
    df_clean = df_clean.drop(columns=['Index'])

# Estandarizar cabeceras (Minúsculas, sin espacios en blanco)
df_clean.columns = df_clean.columns.str.strip().str.lower().str.replace(' ', '_')

# B. Normalización de Tipos de Texto
# Evitar que "Solar", "SOLAR" y "Solar " sean interpretados como tecnologías distintas.
text_cols = ['nombre_proyecto', 'tipo_tecnologia', 'estado_proyecto', 'tipo_proyecto', 'municipio', 'departamento']
for col in text_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip().str.upper()
    df_clean[col] = df_clean[col].replace('NAN', np.nan)

# C. Casteo de Series de Tiempo
# Forzar las variables a datetime reales para permitir operaciones de tiempo (ej. calcular retrasos).
df_clean['fecha_entrada_operacion'] = pd.to_datetime(df_clean['fecha_entrada_operacion'], errors='coerce')
df_clean['fecha_actualizacion'] = pd.to_datetime(df_clean['fecha_actualizacion'], errors='coerce')

# D. Manejo de Nulos e Imputación
# Un proyecto sin Capacidad (MW) o sin Tecnología no aporta a la matriz energética.
df_clean = df_clean.dropna(subset=['capacidad_mw', 'tipo_tecnologia']) 
df_clean['municipio'] = df_clean['municipio'].fillna('SIN REGISTRAR')

# E. Deduplicación final de registros transaccionales
df_clean = df_clean.drop_duplicates()

Index                      0
Nombre Proyecto            0
Capacidad Mw               0
Tipo Tecnologia            0
Estado Proyecto            0
Tipo Proyecto              0
Fecha Entrada Operacion    1
Municipio                  0
Departamento               0
Divipola Municipio         1
Divipola Departamento      0
Fuente                     0
Fecha Actualizacion        0
dtype: int64
0
DataFrame Original:
       Index                      Nombre Proyecto  Capacidad Mw  \
0          1                              GUAYEPO    370.000000   
1          2           PARQUE SOLAR PUERTA DE ORO    300.000000   
2          3                          GUAYEPO III    200.000000   
3          4                            ATLANTICO    180.000000   
4          5                           SHANGRI LA    160.000000   
...      ...                                  ...           ...   
29349  29350  C y C SOLUCIONES EMPRESARIALES LTDA      0.000200   
29350  29351                Maria Cristian Lesmes   

## Comentarios Breves sobre Problemas Detectados
1. Redundancia de Índices: El archivo original incluye una columna Index incrustada, la cual no aporta valor analítico y consume memoria innecesaria al duplicar la función del índice nativo de los DataFrames.

2. Inconsistencias Categóricas: La falta de estandarización en las entradas de texto (mayúsculas/minúsculas y espacios invisibles) generaba particiones erróneas en los agrupamientos geográficos y tecnológicos.

3. Vacíos Críticos de Tiempo: Se detectó un valor nulo en Fecha Entrada Operacion y en Divipola Municipio. Las fechas fueron forzadas a objetos datetime64[ns], dejando las corruptas como nulos (NaT) para evitar fallos durante el modelado de series temporales.

4. Casteo Residual: Al iterar sobre columnas de texto, los valores vacíos de numpy (np.nan) tienden a convertirse en la cadena de caracteres "NAN", un falso positivo que se tuvo que limpiar explícitamente y devolver a nulo para no afectar métricas descriptivas.